<a href="https://colab.research.google.com/github/El-amin/supervised-adversarial-adaptation-model-ADA-/blob/main/ADA_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Supervised ADA (Adversarial Discriminative Adaptation)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models

In [2]:
!pip install kaggle
from google.colab import files
files.upload()


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"aminumusa","key":"6b41d56e89cb10ccd2abcfe82adb7fe5"}'}

In [3]:
# prompt: upload my dataset from kaggle "https://www.kaggle.com/datasets/aminumusa/nigeria-chest-x-ray-dataset"

!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d aminumusa/nigeria-chest-x-ray-dataset
!unzip nigeria-chest-x-ray-dataset.zip


Dataset URL: https://www.kaggle.com/datasets/aminumusa/nigeria-chest-x-ray-dataset
License(s): CC-BY-NC-SA-4.0
Archive:  nigeria-chest-x-ray-dataset.zip
  inflating: my_dataset/test_folder/COVID/COVID-501.png  
  inflating: my_dataset/test_folder/COVID/COVID-502.png  
  inflating: my_dataset/test_folder/COVID/COVID-503.png  
  inflating: my_dataset/test_folder/COVID/COVID-504.png  
  inflating: my_dataset/test_folder/COVID/COVID-505.png  
  inflating: my_dataset/test_folder/COVID/COVID-506.png  
  inflating: my_dataset/test_folder/COVID/COVID-507.png  
  inflating: my_dataset/test_folder/COVID/COVID-508.png  
  inflating: my_dataset/test_folder/COVID/COVID-509.png  
  inflating: my_dataset/test_folder/COVID/COVID-510.png  
  inflating: my_dataset/test_folder/COVID/COVID-511.png  
  inflating: my_dataset/test_folder/COVID/COVID-512.png  
  inflating: my_dataset/test_folder/COVID/COVID-513.png  
  inflating: my_dataset/test_folder/COVID/COVID-514.png  
  inflating: my_dataset/test_folder

In [4]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # Normalize with mean and std of ImageNet
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Paths to your datasets
source_data_path = '/content/my_dataset/train_folder'
target_data_path = '/content/my_dataset/test_folder'

# Create datasets
source_dataset = datasets.ImageFolder(root=source_data_path, transform=transform)
target_dataset = datasets.ImageFolder(root=target_data_path, transform=transform)

# Create DataLoaders
source_loader = DataLoader(source_dataset, batch_size=32, shuffle=True, num_workers=4)
target_loader = DataLoader(target_dataset, batch_size=32, shuffle=True, num_workers=4)


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [5]:
source_loader.dataset.classes

['COVID', 'NORMAL', 'PNEUMONIA', 'TB']

In [6]:


# Feature Extractor (Based on pre-trained ResNet)
class FeatureExtractor(nn.Module):
    def __init__(self):
        super(FeatureExtractor, self).__init__()
        resnet = models.resnet18(pretrained=True)
        self.feature = nn.Sequential(*list(resnet.children())[:-1])  # Remove FC layer

    def forward(self, x):
        x = self.feature(x)
        return x.view(x.size(0), -1)

# Classifier
class Classifier(nn.Module):
    def __init__(self, input_dim=512, num_classes=4):
        super(Classifier, self).__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.fc(x)

# Domain Discriminator
class DomainDiscriminator(nn.Module):
    def __init__(self, input_dim=512):
        super(DomainDiscriminator, self).__init__()
        self.layer = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 2)  # Domain: source or target
        )

    def forward(self, x):
        return self.layer(x)

# Loss functions
criterion_cls = nn.CrossEntropyLoss()
criterion_adv = nn.CrossEntropyLoss()

def compute_adversarial_loss(D, source_feat, target_feat):
    domain_src = torch.zeros(source_feat.size(0)).long().to(source_feat.device)
    domain_tgt = torch.ones(target_feat.size(0)).long().to(target_feat.device)
    domain_preds = torch.cat([D(source_feat), D(target_feat)], dim=0)
    domain_labels = torch.cat([domain_src, domain_tgt], dim=0)
    return criterion_adv(domain_preds, domain_labels)



In [7]:
# Training supervised ADA

def train_supervised_ada(Fs, Ft, C, D, source_loader, target_loader, optimizer_FC, optimizer_D, lambda_adv=1.0):
    Fs.train()
    Ft.train()
    C.train()
    D.train()

    total_loss = 0
    for (src_x, src_y), (tgt_x, tgt_y) in zip(source_loader, target_loader):
        src_x, src_y = src_x.cuda(), src_y.cuda()
        tgt_x, tgt_y = tgt_x.cuda(), tgt_y.cuda()

        # Step 1: Train classifier on both source and target
        src_feat = Fs(src_x)
        tgt_feat = Ft(tgt_x)

        src_pred = C(src_feat)
        tgt_pred = C(tgt_feat)

        loss_src = criterion_cls(src_pred, src_y)
        loss_tgt = criterion_cls(tgt_pred, tgt_y)

        # Step 2: Train domain discriminator
        optimizer_D.zero_grad()
        loss_adv = compute_adversarial_loss(D, src_feat.detach(), tgt_feat.detach())
        loss_adv.backward()
        optimizer_D.step()

        # Step 3: Update feature extractors and classifier
        optimizer_FC.zero_grad()
        fool_loss = criterion_adv(D(tgt_feat), torch.zeros(tgt_feat.size(0)).long().to(tgt_feat.device))
        total = loss_src + loss_tgt + lambda_adv * fool_loss
        total.backward()
        optimizer_FC.step()

        total_loss += total.item()

    return total_loss / len(source_loader)


In [8]:
# Initialize models
Fs = FeatureExtractor().cuda()  # Source feature extractor
Ft = FeatureExtractor().cuda()  # Target feature extractor
C = Classifier().cuda()         # Shared classifier
D = DomainDiscriminator().cuda()  # Domain discriminator

# Optimizers
optimizer_FC = torch.optim.Adam(list(Fs.parameters()) + list(Ft.parameters()) + list(C.parameters()), lr=1e-4)
optimizer_D = torch.optim.Adam(D.parameters(), lr=1e-4)

# Train the model
train_supervised_ada(Fs, Ft, C, D, source_loader, target_loader, optimizer_FC, optimizer_D, lambda_adv=1.0)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 181MB/s]


0.5321150166647775

In [ ]:
import matplotlib.pyplot as plt

# Lists to store accuracy per epoch
source_acc_list = []
target_acc_list = []
loss_list = []

num_epochs = 10
for epoch in range(num_epochs):
    loss = train_supervised_ada(Fs, Ft, C, D, source_loader, target_loader, optimizer_FC, optimizer_D, lambda_adv=1.0)
    loss_list.append(loss)

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss:.4f}")

    print("Evaluation on source domain:")
    src_acc = evaluate_model(Fs, C, source_loader)
    source_acc_list.append(src_acc)

    print("Evaluation on target domain:")
    tgt_acc = evaluate_model(Ft, C, target_loader)
    target_acc_list.append(tgt_acc)

    print("-" * 50)



/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch [1/10], Loss: 0.1857
Evaluation on source domain:
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
Evaluation on target domain:


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
--------------------------------------------------


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import torch.nn.functional as F

def evaluate_model(F_model, C_model, data_loader):
    F_model.eval()
    C_model.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.cuda(), labels.cuda()
            features = F_model(inputs)
            outputs = C_model(features)

            probs = F.softmax(outputs, dim=1)  # Probabilities for all classes
            preds = torch.argmax(outputs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy()[:, 1])  # Keep only probability of class 1 for AUC

    accuracy = accuracy_score(all_labels, all_preds)
    # Changed to 'weighted' for multi-class
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    # Use a suitable method for multi-class AUC if needed (e.g., 'weighted')
    # This is not straightforward with roc_auc_score
    # Further research required here based on your specific needs.
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
    except ValueError:
        auc = "Not calculated (ROC AUC undefined for this case)"

    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")


    return accuracy, precision, recall, auc

In [20]:
evaluate_model(Ft, C, target_loader)

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Accuracy:  0.9750
Precision: 0.9757
Recall:    0.9750


(0.975,
 0.9756663292847503,
 0.975,
 'Not calculated (ROC AUC undefined for this case)')

In [1]:
# Plotting Accuracy
plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs+1), source_acc_list, label='Source Accuracy')
plt.plot(range(1, num_epochs+1), target_acc_list, label='Target Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy Curve over Epochs')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


NameError: name 'plt' is not defined

In [ ]:
# Plotting Loss Curve
plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs+1), loss_list, color='red', label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Curve')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
